# Анализ и прогнозирование оттока клиентов

В этом проекте я анализирую отток клиентов телеком-компании и пробую построить модель, которая сможет заранее выделять клиентов с повышенным риском ухода.

Сначала посмотрю на основные зависимости в данных, затем сравню две модели — логистическую регрессию и случайный лес.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    precision_recall_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

## Загрузка данных

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

In [ ]:
print("Размер датасета:", df.shape)
df.info()

В `TotalCharges` есть пустые значения, из-за которых столбец читается как текст.  
Переведу его в числовой формат. Пустые значения относятся к клиентам с нулевым стажем, поэтому заменю их на 0.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Пропусков в TotalCharges:", df["TotalCharges"].isna().sum())

df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)

## Общая доля оттока

In [ ]:
churn_rate = df["ChurnFlag"].mean()

print(f"Доля клиентов, которые ушли: {churn_rate:.1%}")

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Распределение клиентов по оттоку")
plt.xlabel("")
plt.ylabel("Количество клиентов")
plt.show()

Классы несбалансированы: ушедших клиентов заметно меньше, чем оставшихся.  
Поэтому дальше я не буду ориентироваться только на accuracy — для такой задачи важнее смотреть на ROC-AUC, precision и recall.

## Отток среди пожилых клиентов

In [ ]:
senior_churn = (
    df.groupby("SeniorCitizen")["ChurnFlag"]
      .mean()
      .rename(index={0: "Нет", 1: "Да"})
)

senior_churn

In [ ]:
plt.figure(figsize=(6, 4))
senior_churn.plot(kind="bar")
plt.title("Доля оттока среди пожилых и остальных клиентов")
plt.xlabel("Пожилой клиент")
plt.ylabel("Доля оттока")
plt.xticks(rotation=0)
plt.show()

В этой выборке пожилые клиенты уходят чаще. Сам по себе возраст, конечно, не объясняет причину ухода, но может быть полезным признаком вместе с остальными характеристиками.

## Тип контракта

In [ ]:
contract_churn = (
    df.groupby("Contract")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
)

contract_churn

In [ ]:
plt.figure(figsize=(8, 4))
contract_churn.plot(kind="bar")
plt.title("Доля оттока по типу контракта")
plt.xlabel("")
plt.ylabel("Доля оттока")
plt.xticks(rotation=0)
plt.show()

Сильнее всего выделяются клиенты с помесячным контрактом — среди них отток намного выше, чем среди клиентов с годовыми и двухлетними контрактами.

## Интернет-услуга

In [ ]:
internet_churn = (
    df.groupby("InternetService")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
)

internet_churn

In [ ]:
plt.figure(figsize=(7, 4))
internet_churn.plot(kind="bar")
plt.title("Доля оттока по типу интернет-услуги")
plt.xlabel("")
plt.ylabel("Доля оттока")
plt.xticks(rotation=0)
plt.show()

В этой выборке у клиентов с Fiber optic отток выше, чем у пользователей DSL.

## Количество подключённых дополнительных услуг

In [ ]:
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["NumServices"] = (df[service_cols] == "Yes").sum(axis=1)

services_churn = (
    df.groupby("NumServices")["ChurnFlag"]
      .agg(["mean", "count"])
      .rename(columns={"mean": "churn_rate", "count": "customers"})
)

services_churn

In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(
    x=services_churn.index,
    y=services_churn["churn_rate"],
    marker="o"
)
plt.title("Отток и количество дополнительных услуг")
plt.xlabel("Количество дополнительных услуг")
plt.ylabel("Доля оттока")
plt.show()

В целом отток снижается по мере роста количества подключённых услуг. Это согласуется с идеей, что более вовлечённые клиенты реже уходят.

Теперь перейду от описательного анализа к прогнозированию.

## Подготовка данных для моделей

`customerID` удаляю: это просто идентификатор клиента, полезной информации для прогноза он не несёт.

Данные делю на три части:

- train — обучение;
- validation — сравнение моделей и выбор порога;
- test — финальная проверка.

Разбиение делаю со `stratify`, чтобы доля оттока была примерно одинаковой во всех трёх выборках.

In [ ]:
X = df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df["ChurnFlag"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

validation_share = 0.15 / 0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_share,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

pd.DataFrame({
    "Выборка": ["train", "validation", "test"],
    "Количество строк": [len(X_train), len(X_val), len(X_test)],
    "Доля оттока": [y_train.mean(), y_val.mean(), y_test.mean()]
})

## Preprocessing

In [ ]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "NumServices"
]

categorical_features = [
    col for col in X.columns
    if col not in numeric_features
]

preprocessor = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        numeric_features
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore", drop="if_binary"),
        categorical_features
    )
])

Preprocessing помещаю внутрь `Pipeline`, чтобы масштабирование и кодирование категорий обучались только на train и не было утечки данных.

## Логистическая регрессия

In [ ]:
logreg = Pipeline([
    ("preprocess", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

logreg.fit(X_train, y_train)

## Случайный лес

In [ ]:
rf = Pipeline([
    ("preprocess", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=10,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

rf.fit(X_train, y_train)

## Сравнение моделей

In [ ]:
def get_metrics(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "Precision": precision_score(y_true, predictions),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions)
    }


models = {
    "Logistic Regression": logreg,
    "Random Forest": rf
}

validation_results = []
validation_probabilities = {}

for name, model in models.items():
    probs = model.predict_proba(X_val)[:, 1]
    validation_probabilities[name] = probs

    row = {"Model": name}
    row.update(get_metrics(y_val, probs))
    validation_results.append(row)

validation_results = pd.DataFrame(validation_results)
validation_results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, probs in validation_probabilities.items():
    RocCurveDisplay.from_predictions(
        y_val,
        probs,
        name=name,
        ax=ax
    )

plt.title("ROC-кривые на validation")
plt.show()

ROC-AUC показывает, насколько хорошо модель в целом ранжирует клиентов по риску оттока.

Но для реальной retention-задачи важен ещё и порог: кого именно считать клиентом с высоким риском.

## Выбор порога

Стандартный порог 0.5 не обязательно лучший.

Если компании важнее не пропустить потенциально уходящих клиентов, можно немного снизить порог и повысить recall. Цена этого — больше ложных срабатываний.

Для примера выберу порог по F2-score на validation. F2 сильнее учитывает recall, чем precision.

In [ ]:
def find_best_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probabilities
    )

    precision = precision[:-1]
    recall = recall[:-1]

    f2 = (
        5 * precision * recall
        / (4 * precision + recall + 1e-12)
    )

    best_idx = np.argmax(f2)

    return {
        "threshold": thresholds[best_idx],
        "precision": precision[best_idx],
        "recall": recall[best_idx],
        "f2": f2[best_idx]
    }


threshold_results = []

for name, probs in validation_probabilities.items():
    result = find_best_threshold(y_val, probs)
    result["Model"] = name
    threshold_results.append(result)

threshold_results = pd.DataFrame(threshold_results)
threshold_results

In [ ]:
best_model_name = (
    validation_results
    .sort_values("ROC-AUC", ascending=False)
    .iloc[0]["Model"]
)

best_threshold = float(
    threshold_results.loc[
        threshold_results["Model"] == best_model_name,
        "threshold"
    ].iloc[0]
)

print("Выбранная модель:", best_model_name)
print("Выбранный порог:", round(best_threshold, 3))

## Финальная проверка на test

In [ ]:
test_results = []

for name, model in models.items():
    probs = model.predict_proba(X_test)[:, 1]

    row = {
        "Model": name,
        "Threshold": 0.5
    }
    row.update(get_metrics(y_test, probs, threshold=0.5))
    test_results.append(row)

best_model = models[best_model_name]
best_test_probs = best_model.predict_proba(X_test)[:, 1]

row = {
    "Model": f"{best_model_name} + выбранный порог",
    "Threshold": best_threshold
}
row.update(
    get_metrics(
        y_test,
        best_test_probs,
        threshold=best_threshold
    )
)

test_results.append(row)

test_results = pd.DataFrame(test_results)
test_results

In [ ]:
best_test_pred = (
    best_test_probs >= best_threshold
).astype(int)

fig, ax = plt.subplots(figsize=(5, 5))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, best_test_pred),
    display_labels=["Останется", "Уйдёт"]
).plot(ax=ax, values_format="d")

plt.title("Матрица ошибок")
plt.show()

Здесь особенно интересно смотреть на false negative — клиентов, которые реально ушли, но модель их не выделила.

Для retention-задачи такие ошибки обычно болезненнее, чем лишний контакт с клиентом, который и так бы остался.

## Какие признаки сильнее всего влияют на прогноз

### Коэффициенты логистической регрессии

In [ ]:
feature_names = (
    logreg
    .named_steps["preprocess"]
    .get_feature_names_out()
)

coefficients = (
    logreg
    .named_steps["model"]
    .coef_[0]
)

logreg_features = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients)
    })
    .sort_values("abs_coefficient", ascending=False)
)

logreg_features.head(15)

### Важность признаков Random Forest

In [ ]:
rf_feature_names = (
    rf
    .named_steps["preprocess"]
    .get_feature_names_out()
)

rf_importance = (
    rf
    .named_steps["model"]
    .feature_importances_
)

rf_features = (
    pd.DataFrame({
        "feature": rf_feature_names,
        "importance": rf_importance
    })
    .sort_values("importance", ascending=False)
)

rf_features.head(15)

Важно: высокая важность признака ещё не означает причинно-следственную связь.

Например, если помесячный контракт хорошо помогает прогнозировать отток, из этого нельзя автоматически делать вывод, что перевод клиента на годовой контракт сам по себе снизит вероятность ухода.

## Итог

В проекте я сначала посмотрел, какие группы клиентов чаще уходят, а затем построил две модели для прогноза оттока.

Что получилось сделать:

- подготовить данные и обработать `TotalCharges`;
- посмотреть отток по нескольким важным сегментам;
- построить Logistic Regression и Random Forest;
- сравнить модели по ROC-AUC, precision, recall и F1;
- отдельно подобрать порог на validation;
- проверить итоговый результат на test;
- посмотреть, какие признаки сильнее всего влияют на прогноз.

Практически такую модель можно использовать для приоритизации клиентов в retention-кампании: сначала выделить группу с повышенным риском, а уже потом проверять эффективность конкретного предложения через A/B-тест.

При этом сам прогноз не отвечает на вопрос, почему клиент уходит и какое действие действительно удержит его — для этого нужен отдельный анализ и эксперимент.